In [ ]:
pip install PuLP

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 97.2 MB/s eta 0:00:00


In [ ]:
pip install pandas

In [ ]:
from pulp import *
import pandas as pd
import pulp

In [ ]:
#Data Setup

#origins with supply
origins = {'South America': 1050, 'Southeast Asia': 30, 'Europe': float('inf'), 'ON (Toronto)': 100}

#destination with demand
destinations = {'Calgary': 171, 'Winnipeg': 64, 'Ottawa': 127, 'Montreal': 184, 'Halifax': 64, 'ON (Toronto)': 381, 'Vancouver': 191}

#Hubs
hubs = ['ON (Toronto)', 'FL (Miami)', 'Vancouver']

#cost: origin -> hub
cost_origin_hub = pd.DataFrame({'ON (Toronto)': [14, 35, 30, 0], 'FL (Miami)': [9, 20, 15, None], 'Vancouver': [20, 35, 35, 15]}
                               , index=['South America', 'Southeast Asia', 'Europe', 'ON (Toronto)'])

#cost: hub -> destination
cost_hub_dest = pd.DataFrame({'Calgary': [14, 10, 7], 'Winnipeg': [13, 10, 8], 'Ottawa': [3.5, 7.5, 14], 'Montreal': [3.5, 7.5, 14], 'Halifax': [4, 10, 22], 'ON (Toronto)': [0, 5, 14], 'Vancouver': [15, 10, 0]}
                             , index=['ON (Toronto)', 'FL (Miami)', 'Vancouver'])

In [ ]:
origins

{'South America': 1050,
 'Southeast Asia': 30,
 'Europe': inf,
 'ON (Toronto)': 100}

In [ ]:
destinations

{'Calgary': 171,
 'Winnipeg': 64,
 'Ottawa': 127,
 'Montreal': 184,
 'Halifax': 64,
 'ON (Toronto)': 381,
 'Vancouver': 191}

In [ ]:
cost_origin_hub

,ON (Toronto),FL (Miami),Vancouver
South America,14,9.0,20
Southeast Asia,35,20.0,35
Europe,30,15.0,35
ON (Toronto),0,NaN,15


In [ ]:
cost_hub_dest

,Calgary,Winnipeg,Ottawa,Montreal,Halifax,ON (Toronto),Vancouver
ON (Toronto),14,13,3.5,3.5,4,0,15
FL (Miami),10,10,7.5,7.5,10,5,10
Vancouver,7,8,14.0,14.0,22,14,0


In [ ]:
#Initialize the model

from pulp.constants import LpMinimize
model = pulp.LpProblem("Transshipment", LpMinimize)

# Decision Variables
x = pulp.LpVariable.dicts("x", (origins, hubs), lowBound=0, cat='Continuous')
y = pulp.LpVariable.dicts("y", (hubs, destinations), lowBound=0, cat='Continuous')

In [ ]:
model += (pulp.lpSum(cost_origin_hub.loc[i, j] * x[i][j] for i in origins for j in hubs if pd.notna(cost_origin_hub.loc[i, j])) + pulp.lpSum(cost_hub_dest.loc[j, k] * y[j][k]
               for j in hubs for k in destinations)
), "Total_Cost"

In [ ]:
# Constraints

# Supply at origins
for i in origins:
    if origins[i] != float('inf'):
        model += pulp.lpSum(x[i][j] for j in hubs if pd.notna(cost_origin_hub.loc[i, j])) <= origins[i], f"Supply_{i}"

# Demand at destinations
for k in destinations:
    model += pulp.lpSum(y[j][k] for j in hubs) == destinations[k], f"Demand_{k}"

# Flow balance at hubs
for j in hubs:
    inflow = pulp.lpSum(x[i][j] for i in origins if pd.notna(cost_origin_hub.loc[i, j]))
    outflow = pulp.lpSum(y[j][k] for k in destinations)
    model += inflow == outflow, f"FlowBalance_{j}"

In [ ]:
#Solve and Display

_ = model.solve(pulp.PULP_CBC_CMD(msg=False))
print("Status:", pulp.LpStatus[model.status])
print("Optimal total weekly cost (CAD):", round(pulp.value(model.objective), 2))


Status: Optimal
Optimal total weekly cost (CAD): 18503.5


In [ ]:
model

Transshipment:
MINIMIZE
15.0*x_Europe_FL_(Miami) + 30*x_Europe_ON_(Toronto) + 35*x_Europe_Vancouver + 15*x_ON_(Toronto)_Vancouver + 9.0*x_South_America_FL_(Miami) + 14*x_South_America_ON_(Toronto) + 20*x_South_America_Vancouver + 20.0*x_Southeast_Asia_FL_(Miami) + 35*x_Southeast_Asia_ON_(Toronto) + 35*x_Southeast_Asia_Vancouver + 10*y_FL_(Miami)_Calgary + 10*y_FL_(Miami)_Halifax + 7.5*y_FL_(Miami)_Montreal + 5*y_FL_(Miami)_ON_(Toronto) + 7.5*y_FL_(Miami)_Ottawa + 10*y_FL_(Miami)_Vancouver + 10*y_FL_(Miami)_Winnipeg + 14*y_ON_(Toronto)_Calgary + 4*y_ON_(Toronto)_Halifax + 3.5*y_ON_(Toronto)_Montreal + 3.5*y_ON_(Toronto)_Ottawa + 15*y_ON_(Toronto)_Vancouver + 13*y_ON_(Toronto)_Winnipeg + 7*y_Vancouver_Calgary + 22*y_Vancouver_Halifax + 14.0*y_Vancouver_Montreal + 14*y_Vancouver_ON_(Toronto) + 14.0*y_Vancouver_Ottawa + 8*y_Vancouver_Winnipeg + 0.0
SUBJECT TO
Supply_South_America: x_South_America_FL_(Miami)
 + x_South_America_ON_(Toronto) + x_South_America_Vancouver <= 1050

Supply_Southea

In [ ]:
# Display all flows with a nonzero value
flows_x = [(i, j, x[i][j].varValue)
           for i in origins for j in hubs
           if x[i][j].varValue is not None]

flows_y = [(j, k, y[j][k].varValue)
           for j in hubs for k in destinations
           if y[j][k].varValue is not None]

print("\n--- Origin -> Hub Flows ---")
for i, j, q in flows_x:
    print(f"{i:15} -> {j:12} : {q:.2f}")

print("\n--- Hub -> Destination Flows ---")
for j, k, q in flows_y:
    print(f"{j:12} -> {k:12} : {q:.2f}")


--- Origin -> Hub Flows ---
South America   -> ON (Toronto) : 0.00
South America   -> FL (Miami)   : 1050.00
South America   -> Vancouver    : 0.00
Southeast Asia  -> ON (Toronto) : 0.00
Southeast Asia  -> FL (Miami)   : 0.00
Southeast Asia  -> Vancouver    : 0.00
Europe          -> ON (Toronto) : 0.00
Europe          -> FL (Miami)   : 32.00
Europe          -> Vancouver    : 0.00
ON (Toronto)    -> ON (Toronto) : 100.00
ON (Toronto)    -> Vancouver    : 0.00

--- Hub -> Destination Flows ---
ON (Toronto) -> Calgary      : 0.00
ON (Toronto) -> Winnipeg     : 0.00
ON (Toronto) -> Ottawa       : 0.00
ON (Toronto) -> Montreal     : 0.00
ON (Toronto) -> Halifax      : 64.00
ON (Toronto) -> ON (Toronto) : 36.00
ON (Toronto) -> Vancouver    : 0.00
FL (Miami)   -> Calgary      : 171.00
FL (Miami)   -> Winnipeg     : 64.00
FL (Miami)   -> Ottawa       : 127.00
FL (Miami)   -> Montreal     : 184.00
FL (Miami)   -> Halifax      : 0.00
FL (Miami)   -> ON (Toronto) : 345.00
FL (Miami)   -> Vancouv

In [ ]:
# Sensitivity Analysis
print()
print("Sensitivity Analysis:")
MF_SA_df = pd.DataFrame([
    {
        'constraint': name,
        'slack': c.slack,
        'shadow price': c.pi
    }
    for name, c in model.constraints.items()
])
print(MF_SA_df)


Sensitivity Analysis:
                  constraint  slack  shadow price
0       Supply_South_America   -0.0          -6.0
1      Supply_Southeast_Asia   30.0           0.0
2        Supply_ON_(Toronto)   -0.0         -20.0
3             Demand_Calgary   -0.0          25.0
4            Demand_Winnipeg   -0.0          25.0
5              Demand_Ottawa   -0.0          22.5
6            Demand_Montreal   -0.0          22.5
7             Demand_Halifax   -0.0          24.0
8        Demand_ON_(Toronto)   -0.0          20.0
9           Demand_Vancouver   -0.0          25.0
10  FlowBalance_ON_(Toronto)   -0.0          20.0
11    FlowBalance_FL_(Miami)   -0.0          15.0
12     FlowBalance_Vancouver   -0.0          25.0
